# 1. Modelo Logistic Regression (statsmodels)

Primer modelo de clasificación binaria para predecir Heart Disease.

**Estrategia:**
1. Modelo 1: Todas las variables
2. Modelo 2: Variables principales según EDA
3. Modelo 3: Solo coeficientes significativos (p < 0.05)

**Evaluación:** ROC-AUC mediante cross-validation (5-fold)

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

## 2. Carga de datos

In [ ]:
df = pd.read_csv("../../data/train.csv")

# Target binario
df["target"] = (df["Heart Disease"] == "Presence").astype(int)

# Features
continuous_features = ["Age", "BP", "Cholesterol", "Max HR", "ST depression"]
categorical_features = ["Sex", "Chest pain type", "FBS over 120", "EKG results",
                        "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium"]
all_features = continuous_features + categorical_features

y = df["target"]
X = df[all_features]

print(f"Shape: {X.shape}")
print(f"Target balance: {y.mean():.2%} Presence")

## 3. Modelo 1 — Todas las variables

In [ ]:
X1 = sm.add_constant(X)
model1 = sm.Logit(y, X1).fit(disp=0)
print(model1.summary())

In [ ]:
# Identificar coeficientes no significativos
pvalues1 = model1.pvalues.drop("const")
non_sig1 = pvalues1[pvalues1 > 0.05]

print("Coeficientes NO significativos (p > 0.05):")
if len(non_sig1) == 0:
    print("  Todos los coeficientes son significativos.")
else:
    for var, p in non_sig1.sort_values(ascending=False).items():
        print(f"  {var}: p = {p:.4f}")

print(f"\nVariables significativas: {len(pvalues1) - len(non_sig1)} / {len(pvalues1)}")

## 4. Modelo 2 — Variables principales del EDA

Se excluyen las variables de baja importancia según el ranking del EDA:
- Age (score: 35.01)
- Cholesterol (score: 13.66)
- FBS over 120 (score: 5.54)
- BP (score: 0.86)

In [ ]:
eda_top_features = ["Thallium", "Chest pain type", "Number of vessels fluro",
                    "Exercise angina", "Max HR", "ST depression",
                    "Slope of ST", "Sex", "EKG results"]

X2 = sm.add_constant(df[eda_top_features])
model2 = sm.Logit(y, X2).fit(disp=0)
print(model2.summary())

In [ ]:
# Identificar coeficientes no significativos
pvalues2 = model2.pvalues.drop("const")
non_sig2 = pvalues2[pvalues2 > 0.05]

print("Coeficientes NO significativos (p > 0.05):")
if len(non_sig2) == 0:
    print("  Todos los coeficientes son significativos.")
else:
    for var, p in non_sig2.sort_values(ascending=False).items():
        print(f"  {var}: p = {p:.4f}")

print(f"\nVariables significativas: {len(pvalues2) - len(non_sig2)} / {len(pvalues2)}")

## 5. Modelo 3 — Solo coeficientes significativos

Backward elimination: se eliminan iterativamente las variables con p > 0.05 hasta que todos los coeficientes sean significativos.

In [ ]:
# Backward elimination desde las variables del EDA
features_remaining = eda_top_features.copy()
iteration = 0

while True:
    iteration += 1
    X_iter = sm.add_constant(df[features_remaining])
    model_iter = sm.Logit(y, X_iter).fit(disp=0)

    pvals = model_iter.pvalues.drop("const")
    max_pval = pvals.max()
    worst_var = pvals.idxmax()

    if max_pval <= 0.05:
        print(f"Iteraci\u00f3n {iteration}: Todos los p-values <= 0.05. Fin.")
        break

    print(f"Iteraci\u00f3n {iteration}: Eliminando '{worst_var}' (p = {max_pval:.4f})")
    features_remaining.remove(worst_var)

    if len(features_remaining) == 0:
        print("No quedan variables.")
        break

# Modelo final
X3 = sm.add_constant(df[features_remaining])
model3 = sm.Logit(y, X3).fit(disp=0)

print(f"\nVariables finales ({len(features_remaining)}): {features_remaining}")
print(model3.summary())

## 6. Evaluaci\u00f3n — ROC-AUC con Cross-Validation (5-Fold)

In [ ]:
def cv_roc_auc(X_df, y_series, n_splits=5, random_state=42):
    """Cross-validation con statsmodels Logit. Retorna scores por fold."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = []
    fpr_list, tpr_list = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_df, y_series), 1):
        X_train = sm.add_constant(X_df.iloc[train_idx])
        X_val = sm.add_constant(X_df.iloc[val_idx])
        y_train = y_series.iloc[train_idx]
        y_val = y_series.iloc[val_idx]

        model = sm.Logit(y_train, X_train).fit(disp=0)
        y_prob = model.predict(X_val)

        auc = roc_auc_score(y_val, y_prob)
        scores.append(auc)

        fpr, tpr, _ = roc_curve(y_val, y_prob)
        fpr_list.append(fpr)
        tpr_list.append(tpr)

    return scores, fpr_list, tpr_list

In [ ]:
# Definir features de cada modelo
models_config = {
    "Modelo 1 (todas)": all_features,
    "Modelo 2 (EDA top)": eda_top_features,
    "Modelo 3 (significativas)": features_remaining,
}

results = {}
roc_data = {}

for name, features in models_config.items():
    print(f"Evaluando {name} ({len(features)} variables)...")
    scores, fpr_list, tpr_list = cv_roc_auc(df[features], y)
    results[name] = scores
    roc_data[name] = (fpr_list, tpr_list)

    mean_auc = np.mean(scores)
    std_auc = np.std(scores)
    print(f"  ROC-AUC: {mean_auc:.4f} (+/- {std_auc:.4f})")
    for i, s in enumerate(scores, 1):
        print(f"    Fold {i}: {s:.4f}")
    print()

## 7. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
colors = ["steelblue", "coral", "seagreen"]

for (name, (fpr_list, tpr_list)), color in zip(roc_data.items(), colors):
    # Graficar la curva del ultimo fold
    mean_auc = np.mean(results[name])
    ax.plot(fpr_list[-1], tpr_list[-1], color=color, linewidth=2,
            label=f"{name} (AUC = {mean_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Curvas ROC - Comparaci\u00f3n de Modelos", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Comparaci\u00f3n de Modelos

In [ ]:
comparison = pd.DataFrame({
    "Modelo": list(results.keys()),
    "N Variables": [len(v) for v in models_config.values()],
    "ROC-AUC Mean": [np.mean(s) for s in results.values()],
    "ROC-AUC Std": [np.std(s) for s in results.values()],
})
comparison["ROC-AUC"] = comparison.apply(
    lambda r: f"{r['ROC-AUC Mean']:.4f} +/- {r['ROC-AUC Std']:.4f}", axis=1
)

print("="*80)
print("COMPARACI\u00d3N DE MODELOS")
print("="*80)
print(comparison[["Modelo", "N Variables", "ROC-AUC"]].to_string(index=False))
print("="*80)

## 9. Modelo Final Seleccionado

In [ ]:
# El modelo seleccionado es el Modelo 3 (solo coeficientes significativos)
print("="*80)
print("MODELO FINAL: Modelo 3 (solo coeficientes significativos)")
print("="*80)
print(f"\nVariables ({len(features_remaining)}):")
for i, var in enumerate(features_remaining, 1):
    print(f"  {i}. {var}")

mean_auc = np.mean(results["Modelo 3 (significativas)"])
std_auc = np.std(results["Modelo 3 (significativas)"])
print(f"\nROC-AUC (5-fold CV): {mean_auc:.4f} +/- {std_auc:.4f}")

print("\nCoeficientes del modelo final:")
coef_df = pd.DataFrame({
    "Variable": model3.params.index,
    "Coef": model3.params.values,
    "p-value": model3.pvalues.values,
})
print(coef_df.to_string(index=False))
print("="*80)